# Standalone SGD neural-network experiment
Ten seeded networks with two ReLU hidden layers (32 and 16 units), trained for
exactly 100 epochs using minibatch SGD (learning rate 0.01, no momentum,
batch size 64) and cross-entropy loss. No early stopping or validation split.

Train on elections through 2015 and evaluate on 2019. The 2017 rows are unused.
Load both CSVs directly; the separate 2024 test data remains unused. All model and
preprocessing code is defined here, with no imports from project modules.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from IPython.display import display

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "TEST_TRAIN" / "train.csv").is_file()
)
train = pd.read_csv(project_root / "TEST_TRAIN" / "train.csv")
test = pd.read_csv(project_root / "TEST_TRAIN" / "test.csv")

FEATURES = [
    "country/region", "previous_majority_proportion", "previous_winner",
    "Conservative", "Labour", "LD", "incumbent", "previous_con_share",
    "previous_lib_share", "previous_lab_share", "previous_natSW_share",
    "projected_con_share", "projected_lib_share", "projected_lab_share",
]
CATEGORICAL = ["country/region", "previous_winner", "incumbent"]
NUMERIC = [column for column in FEATURES if column not in CATEGORICAL]
SEEDS = (29798, 58084, 77167, 85291, 72969, 93189, 8914, 95011, 30096, 63732)
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.01
TRAIN_THROUGH = 2015
EVALUATION_YEAR = 2019

years = pd.to_numeric(train["election"], errors="raise")
training_data = train.loc[years <= TRAIN_THROUGH].copy()
evaluation_data = train.loc[years == EVALUATION_YEAR].dropna(subset=["winner"]).copy()
if training_data.empty or training_data["winner"].isna().any():
    raise ValueError("Training requires nonempty data with known winners.")
if evaluation_data.empty:
    raise ValueError("No evaluation rows with known winners.")
print(f"Training through {TRAIN_THROUGH}: {len(training_data)} rows")
print(f"Evaluation in {EVALUATION_YEAR}: {len(evaluation_data)} rows")
print(f"Separate test CSV loaded but unused: {len(test)} rows")


In [ ]:
# Fit every preprocessing step on training rows only.
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
    ]), NUMERIC),
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__",
                                keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CATEGORICAL),
])

X_train_matrix = np.asarray(
    preprocessor.fit_transform(training_data[FEATURES]), dtype=np.float32
)
X_evaluation_matrix = np.asarray(
    preprocessor.transform(evaluation_data[FEATURES]), dtype=np.float32
)
if not np.isfinite(X_train_matrix).all() or not np.isfinite(X_evaluation_matrix).all():
    raise ValueError("Preprocessing produced non-finite features.")
label_encoder = LabelEncoder().fit(training_data["winner"])
X_train = torch.from_numpy(X_train_matrix)
X_evaluation = torch.from_numpy(X_evaluation_matrix)
y_train = torch.tensor(label_encoder.transform(training_data["winner"]), dtype=torch.long)
training_dataset = TensorDataset(X_train, y_train)
print(f"Training tensor: {tuple(X_train.shape)}; evaluation tensor: {tuple(X_evaluation.shape)}")


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, number_of_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, number_of_classes),
        )

    def forward(self, features):
        return self.layers(features)  # Raw logits for CrossEntropyLoss.


models = []
training_history = []
# Small CPU networks avoid unnecessary thread overhead.
torch.set_num_threads(1)
for seed in SEEDS:
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = NeuralNetwork(X_train.shape[1], len(label_encoder.classes_))
        optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.0)
        criterion = nn.CrossEntropyLoss()
        loader = DataLoader(
            training_dataset, batch_size=BATCH_SIZE, shuffle=True,
            generator=torch.Generator().manual_seed(seed), num_workers=0,
        )
        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            for features, targets in loader:
                optimizer.zero_grad()
                loss = criterion(model(features), targets)
                if not torch.isfinite(loss):
                    raise RuntimeError(f"Non-finite loss: seed {seed}, epoch {epoch}")
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(targets)
            training_history.append({
                "seed": seed, "epoch": epoch,
                "training_loss": total_loss / len(training_dataset),
            })
        model.eval()
        models.append(model)
        print(f"Seed {seed}: completed {EPOCHS} epochs; training loss {training_history[-1]['training_loss']:.4f}")
training_history = pd.DataFrame(training_history)


In [ ]:
# Evaluate only after all ten fixed-duration fits have finished.
if len(models) != len(SEEDS):
    raise ValueError("Train all ten models before evaluation.")
probabilities_by_model = []
individual_scores = []
with torch.inference_mode():
    for seed, model in zip(SEEDS, models):
        model.eval()
        probabilities = torch.softmax(model(X_evaluation), dim=1).numpy()
        probabilities_by_model.append(probabilities)
        predictions = label_encoder.inverse_transform(probabilities.argmax(axis=1))
        individual_scores.append({
            "seed": seed,
            "accuracy": accuracy_score(evaluation_data["winner"], predictions),
        })
display(pd.DataFrame(individual_scores))

ensemble_probabilities = np.mean(probabilities_by_model, axis=0)
ensemble_predictions = label_encoder.inverse_transform(ensemble_probabilities.argmax(axis=1))
ensemble_accuracy = accuracy_score(evaluation_data["winner"], ensemble_predictions)
print(f"{EVALUATION_YEAR} ensemble accuracy: {ensemble_accuracy:.2%} ({len(evaluation_data)} rows)")
complete = evaluation_data[FEATURES].notna().all(axis=1)
if complete.any():
    complete_accuracy = accuracy_score(
        evaluation_data.loc[complete, "winner"], ensemble_predictions[complete.to_numpy()]
    )
    print(f"Complete-row accuracy: {complete_accuracy:.2%} ({complete.sum()} rows)")
